# Comparación final — Cuatro enfoques y baseline

Este notebook **no vuelve a ajustar hiperparámetros**. Reúne los resultados de test ya generados por los cuatro notebooks con el mismo protocolo. Usar una misma métrica no permite mezclar muestras diferentes.

**Antes de ejecutar:** terminar los notebooks 01 a 04; mantener N_VALIDACION, N_TEST y SEMILLA idénticos. El antiguo TF-IDF + K-Means se conserva aparte como experimento exploratorio; su Silhouette no entra en esta tabla de recomendación.

In [ ]:
from pathlib import Path
import sys
import heapq
from itertools import islice
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

# El archivo sephora_common.py debe estar junto a los notebooks.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "data" / "processed").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Abrí este notebook dentro del proyecto que contiene data/processed")
helper_dir = next((p for p in [Path.cwd(), PROJECT_ROOT / "notebooks"]
                   if (p / "sephora_common.py").exists()), None)
if helper_dir is None:
    raise FileNotFoundError("Copiá sephora_common.py a notebooks/ junto a estos archivos")
sys.path.insert(0, str(helper_dir))
import sephora_common as sc
from sephora_common import PROFILE_COLS, rank_scores, text_value

# Los mismos valores en los cuatro notebooks.
N_VALIDACION = 500
N_TEST = 500
SEMILLA = 42
ctx = sc.load_context(N_VALIDACION, N_TEST, SEMILLA)
print("Protocolo:", ctx.protocol_id[:12])
print("Resultados:", ctx.output_dir)
display(pd.Series({k:v for k,v in ctx.manifest.items()
                   if k not in ["validation_cases", "test_cases", "versions"]}).to_frame("valor"))
display(pd.DataFrame(ctx.manifest["versions"], index=[0]))

## 2. Condiciones comunes de comparación

- **Consulta:** tipo y tono de piel, ojos y cabello. Cada enfoque decide qué atributos utiliza. Ninguno recibe compras ni ratings del usuario de consulta.
- **Candidatos:** catálogo Skincare, idéntico para todos. Un modelo basado en evidencia puede puntuar solo parte de este catálogo; ese límite se refleja en la cobertura.
- **Positivo:** `rating >= 4` **o** `is_recommended = True`. La misma regla sirve para todos. Una review de rating bajo pero recomendada es positiva según esta definición.
- **Datos:** un par usuario–producto por vez; se conserva la última review con fecha disponible. Se cuantifica lo descartado. Los perfiles se resumen por moda, sin utilizar ratings para definirlos.
- **Validación:** 500 usuarios para elegir hiperparámetros.
- **Test:** otros 500 usuarios para evaluar la configuración elegida. Se fija un producto positivo por usuario.
- **Sin fuga:** ninguna review de esos 1.000 usuarios entra en entrenamiento, estadísticas de ratings, popularidad o votos.
- **Cold start:** los demás positivos del usuario también permanecen ocultos. No se utilizan ni siquiera para filtrar el Top-10.

Se seleccionan usuarios con al menos dos productos positivos y tipo/tono conocidos. Es una evaluación en esa población, no en todos los visitantes del sitio. Se supone que los atributos personales aportados en las reviews son conocidos al consultar. Esta partición no simula el paso del tiempo.

La huella del protocolo verifica que datos, semilla y tamaños sean iguales. Si cambian, los resultados van a otra carpeta; el comparador no mezcla protocolos.

## 3. Tabla comparativa verificada

In [ ]:
import json
# Deben existir cuatro resultados de ESTE protocolo.
NOMBRES = ["01_vecinos_por_perfil", "02_satisfaccion_por_segmento",
           "03_similitud_ponderada", "04_afinidad_supervisada"]
resultados = []
for nombre in NOMBRES:
    path = ctx.output_dir / f"{nombre}.json"
    if not path.exists():
        raise FileNotFoundError(f"Falta {path.name}. Ejecutá y guardá ese notebook con el mismo protocolo.")
    r = json.loads(path.read_text(encoding="utf-8"))
    if r["protocol_id"] != ctx.protocol_id:
        raise ValueError("Resultados de protocolos distintos")
    resultados.append(r)
# Baseline tiene que coincidir en las cuatro ejecuciones (excepto tiempos).
campos = ["users", "hits", "hit_rate_at_10", "ndcg_at_10", "user_coverage", "catalog_coverage"]
for r in resultados[1:]:
    for campo in campos:
        if not np.isclose(r["baseline_test"][campo], resultados[0]["baseline_test"][campo]):
            raise ValueError(f"Baseline inconsistente en {campo}")
filas = [{"modelo": "Baseline global", "config": "Popularidad positiva", **resultados[0]["baseline_test"]}]
filas += [{"modelo": r["model"], "config": json.dumps(r["selected_config"]), **r["test"]} for r in resultados]
tabla = pd.DataFrame(filas).set_index("modelo")
display(tabla)
tabla.to_csv(ctx.output_dir / "comparacion_final.csv")
print("Tabla exportada a:", ctx.output_dir / "comparacion_final.csv")

## 4. Incertidumbre de la diferencia respecto al baseline

Se estima un intervalo orientativo mediante bootstrap pareado de usuarios. Es un análisis exploratorio, no una garantía de superioridad ni una corrección por todas las comparaciones realizadas. Una muestra de 500 usuarios puede producir diferencias pequeñas e inestables.

In [ ]:
# Diferencia de HR sobre los mismos usuarios: bootstrap pareado exploratorio.
# Baseline se recalcula con las reglas comunes, no se reutiliza una cifra vieja.
_, detalle_baseline = sc.evaluate(sc.PopularityBaseline(ctx), ctx, ctx.test, {}, False)
rng = np.random.default_rng(2026)
intervalos = []
for r in resultados:
    detail = pd.read_csv(ctx.output_dir / f"{r['model']}_test_detail.csv", dtype={"author_id": str, "target": str})
    paired = detail.merge(detalle_baseline[["author_id", "target", "hit"]],
                          on=["author_id", "target"], suffixes=("_modelo", "_baseline"), validate="one_to_one")
    if len(paired) != len(ctx.test):
        raise ValueError("No coinciden los casos de test")
    dif = (paired.hit_modelo-paired.hit_baseline).to_numpy()
    draws = rng.choice(dif, size=(2000, len(dif)), replace=True).mean(axis=1)
    low, high = np.quantile(draws, [0.025, 0.975])
    intervalos.append({"modelo": r["model"], "diferencia_hr_pp": 100*dif.mean(),
                       "IC95_inferior_pp": 100*low, "IC95_superior_pp": 100*high})
display(pd.DataFrame(intervalos))
print("Si el intervalo incluye cero, esta muestra no separa claramente al modelo del baseline.")

## 5. Ejemplos para un mismo perfil

Todos utilizan piel oily, tono dark, ojos blue y cabello brown. Comparar nombres, categorías, cantidad de productos y coincidencias. Una menor coincidencia con el baseline indica una salida diferente, no que sea mejor.

In [ ]:
for name in NOMBRES:
    print("\n", name)
    example = pd.read_csv(ctx.output_dir / f"{name}_example_top10.csv")
    display(example)
# Coincidencia con popularidad para el mismo perfil; describe diferenciación.
perfil = {"skin_type": "oily", "skin_tone": "dark", "eye_color": "blue", "hair_color": "brown"}
ids_global = set(sc.PopularityBaseline(ctx).recommend(perfil, {}))
coincidencias = []
for name in NOMBRES:
    ids = set(pd.read_csv(ctx.output_dir / f"{name}_example_top10.csv").product_id)
    coincidencias.append({"modelo": name, "productos_comunes_baseline": len(ids & ids_global),
                         "coincidencia_sobre_10": len(ids & ids_global)/10})
display(pd.DataFrame(coincidencias))

## 6. Justificación del modelo elegido

La mayor métrica es un resultado, no una justificación completa. Completar después de ejecutar:

- **Modelo elegido:** …
- **Objetivo:** recomendar skincare para una persona que aporta su perfil sin historial.
- **Resultados de test:** HR@10 …; NDCG@10 …; cobertura de usuarios …; catálogo ….
- **Comparación con baseline:** diferencia …; incertidumbre ….
- **Motivo de elección:** relevancia observada, cobertura, simplicidad, posibilidad de explicación y costo.
- **Alternativas:** qué aporta cada una y por qué no se selecciona.
- **Limitaciones:** sesgo de quienes reseñan, ratings mayormente positivos, perfil incompleto, productos sin interacciones y muestra restringida a usuarios con tipo/tono conocidos.

Si ningún modelo supera al baseline, decirlo. Puede conservarse un prototipo personalizado como demostración, pero no afirmar que mejora la calidad de recomendación. El perfil no garantiza eficacia dermatológica.

**No ajustar nuevamente al mirar test:** eso convertiría test en otra validación. Los cambios posteriores requieren otra evaluación independiente.